# High-Injury Network Analysis

A common roadway safety analysis pattern involves the development of a High-Injury Network (HIN). One typical approach to HIN development uses a sliding window-style analysis to create a profile of crash risk along roadway corridors based on historical crash data. Because crashes are point events and the roadway data we are evaluating is linear, it is necessary to find an effective way to smooth that data to generalize crash patterns. This can be done in a variety of ways, such as using the `distribute` aggregator on an events relationship between a resegmented roadway dataset and a crash dataset. The example below uses the provided sample `roadway` and `crashes` datasets to create a simple HIN using this methodology.

This notebook is self-contained and does not depend on other example notebooks.

## Load Data

First, let's set our default LRS and load our datasets:

In [ ]:
import linref as lr

# Set the default LRS
lr.set_default_lrs(
    key_col=['route'],
    chain_col='chain',
    loc_col='loc',
    beg_col='beg',
    end_col='end',
    geom_col='geometry',
    geom_m_col='geometry_m',
    closed='left_mod'
)

# Load built-in datasets
roadways = lr.datasets.load('roadways')
crashes  = lr.datasets.load('crashes')

print(f'Miles of roadways: {roadways.lr.event_lengths.sum():.2f}')
print(f'Number of crashes: {len(crashes)}')

Miles of roadways: 32.30
Number of crashes: 20


## Resegment Roadways

Next, we need to resegment our roadways using a standard segment length. This will define the unit of our analysis. Typically, we will use a segment length between 0.1 and 1.0 miles, depending on the context and goals of the analysis as well as the density of crashes in the study area. Segment lengths may be longer in less dense rural areas or shorter in more dense urban areas. For this example, let's use a segment length of 0.5 miles. To avoid particularly short segments at the ends of corridors, we will use the `fill='balance'` parameter.

In [2]:
# First, dissolve the roadways to create continuous segments
dissolved = roadways.lr.dissolve()

# Resegment the roadways to a standard length
resegmented = dissolved.lr.resegment(length=0.5, fill='balance')

print(f'Number of original segments: {len(roadways)}')
print(f'Number of dissolved segments: {len(dissolved)}')
print(f'Number of resegmented segments: {len(resegmented)}')

Number of original segments: 10
Number of dissolved segments: 3
Number of resegmented segments: 65


## Crash Data Distribution

Now we are ready to perform the analysis. We will relate the crashes data to the resegmented roadways layer and apply the `distribute` aggregator. This method identifies the segment that the crash falls on and then distributes the value of the crash between that segment and a number of adjacent segments, effectively smoothing the data along each corridor. Exactly how this distribution is done can be modified using a variety of parameters that effect the relative proportion of a crash that is assigned to the initial segment and adjacent segments, decreasing with greater distance.

Here, we will use standard parameters with `decay_func='linear'` and `decay_size=2`, distributing the value of each crash between the segment that it occurred on and two segments on either side. For example, assume that our roadway is segmented at 0.5 mile intervals and a crash occurs on a given route at milepost 1.2. Using these parameters, the distribute aggregator will apply 0.333 of that crash to the [1.0, 1.5) segment, 0.222 to each of the [0.5, 1.0) and [1.5, 2.0) segments, and 0.111 to each of the [0.0, 0.5) and [2.0, 2.5) segments. Note that edge cases will have slightly higher overall weights due to the nature of this approach. For example, a crash occurring at milepost 0.6 will be distributed with a value of 0.25 on the [0.0, 0.5) segment, 0.375 on the [0.5, 1.0) segment, 0.25 on the [1.0, 1.5) segment, and 0.125 on the [1.5, 2.0) segment.

In [3]:
# Create a relation between crashes and resegmented roadways
relation = resegmented.lr.relate(crashes)

# Perform crash distribution
resegmented['crash_score'] = relation.distribute(
    decay_func='linear',
    decay_size=2
)

# For comparison, also compute simple crash counts
resegmented['crash_counts'] = relation.count()

print(f'Total crash score: {resegmented["crash_score"].sum():.2f}')
print(f'Total crash counts: {resegmented["crash_counts"].sum():.2f}')
print(resegmented.lr.get_group('I-5').iloc[5:14][['route', 'beg', 'end', 'crash_score', 'crash_counts']])

Total crash score: 20.00
Total crash counts: 20.00
   route  beg  end  crash_score  crash_counts
5    I-5  2.5  3.0     0.000000             0
6    I-5  3.0  3.5     0.111111             0
7    I-5  3.5  4.0     0.222222             0
8    I-5  4.0  4.5     0.333333             1
9    I-5  4.5  5.0     0.333333             0
10   I-5  5.0  5.5     0.333333             0
11   I-5  5.5  6.0     0.333333             1
12   I-5  6.0  6.5     0.222222             0
13   I-5  6.5  7.0     0.111111             0


Consider the results of this analysis: though the `crash_score` column contains decimal values, it still sums to a number equal to the count of total crashes being analyzed. Because of this, we can still summarize the results in terms of actual crash counts instead of a unitless index which may be beneficial for some applications.